# Bronze pipeline del parcial

Este notebook explica el pipeline de ingesta Bronze del parcial para las tres fuentes: `ingresos`, `sismepre` y `renamu`. El objetivo es dejar trazabilidad clara del flujo, cómo se lee `config.yaml`, cómo se ejecuta el pipeline y cómo se verifica la evidencia en `data/bronze` y `data/audit`.

In [1]:
from pathlib import Path
import sys
import json
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze'
AUDIT_PATH = PROJECT_ROOT / 'data' / 'audit'
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as file_handle:
    config = yaml.safe_load(file_handle)

print(f'Project root: {PROJECT_ROOT}')
print(f'Bronze path: {BRONZE_PATH}')
print(f'Audit path: {AUDIT_PATH}')

Project root: /home/jovyan/work
Bronze path: /home/jovyan/work/data/bronze
Audit path: /home/jovyan/work/data/audit


## Fuentes del parcial

En esta práctica el pipeline Bronze trabaja con tres fuentes:

1. **SIAF Ingresos**: mezcla de archivos históricos y recursos API.
2. **SISMEPRE**: recursos del impuesto predial expuestos como `datastore_search`.
3. **RENAMU**: descarga directa del PDF de diccionario y del ZIP de datos 2022.

La capa Bronze mantiene los datos en estado crudo o casi crudo, sin aplicar transformaciones analíticas ni construir Silver o Gold.

In [2]:
resource_rows = []
for dataset_name, dataset_config in config['datasets'].items():
    if dataset_name == 'ingresos':
        resource_rows.append({'dataset': dataset_name, 'group': 'diccionario', 'filename': dataset_config['diccionario']['filename'], 'url': dataset_config['diccionario']['url']})
        for item in dataset_config.get('historico', []):
            resource_rows.append({'dataset': dataset_name, 'group': 'historico', 'filename': item['filename'], 'url': item['url']})
        for item in dataset_config.get('api', []):
            resource_rows.append({'dataset': dataset_name, 'group': 'api', 'filename': item['filename'], 'url': item['url']})
    elif dataset_name == 'sismepre':
        for group_name in ['diccionarios', 'archivos']:
            for item in dataset_config.get(group_name, []):
                resource_rows.append({'dataset': dataset_name, 'group': group_name, 'filename': item['filename'], 'url': item['url']})
    elif dataset_name == 'renamu':
        for group_name, item in dataset_config.items():
            resource_rows.append({'dataset': dataset_name, 'group': group_name, 'filename': item['filename'], 'url': item['url']})

resources_df = pd.DataFrame(resource_rows)
resources_df

,dataset,group,filename,url
0,ingresos,diccionario,Ingresos_Diccionario.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
1,ingresos,historico,2012-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
2,ingresos,historico,2013-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
3,ingresos,historico,2014-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
4,ingresos,historico,2015-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
5,ingresos,historico,2016-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
6,ingresos,historico,2017-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
7,ingresos,historico,2018-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
8,ingresos,historico,2019-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...
9,ingresos,historico,2020-Ingreso.csv,https://fs.datosabiertos.mef.gob.pe/datastoref...


## Arquitectura Bronze

El pipeline se implementó siguiendo la idea del manual del BCR, pero adaptado al parcial. La secuencia es `config -> clients -> storage -> services -> pipelines -> audit`.

- `settings.py` carga `config.yaml` y define rutas de Bronze y Audit.
- `download_client.py` descarga archivos directos.
- `mef_client.py` pagina la API `datastore_search`.
- `data_lake.py` escribe archivos CSV en `data/bronze/<dataset>`.
- `ingestion_service.py` decide si un recurso va por API o por descarga directa.
- `ingresos_pipeline.py`, `sismepre_pipeline.py` y `renamu_pipeline.py` ejecutan la ingesta por fuente.
- `bronze_pipeline.py` orquesta todo y registra el resultado en `data/audit`.

In [3]:
architecture_rows = [
    {'nivel': 'Config', 'modulo': 'app/config/settings.py', 'responsabilidad': 'Lee config.yaml y expone datasets y rutas'},
    {'nivel': 'Clients', 'modulo': 'app/clients/download_client.py', 'responsabilidad': 'Descarga archivos directos'},
    {'nivel': 'Clients', 'modulo': 'app/clients/mef_client.py', 'responsabilidad': 'Pagina recursos MEF expuestos por datastore_search'},
    {'nivel': 'Storage', 'modulo': 'app/storage/data_lake.py', 'responsabilidad': 'Escribe CSV en Bronze'},
    {'nivel': 'Service', 'modulo': 'app/services/ingestion_service.py', 'responsabilidad': 'Coordina descarga, escritura y quality checks'},
    {'nivel': 'Pipelines', 'modulo': 'app/pipelines/bronze/*.py', 'responsabilidad': 'Orquestación por dataset y pipeline maestro'},
    {'nivel': 'Audit', 'modulo': 'app/audit/*.py', 'responsabilidad': 'Registra ejecuciones, quality checks y errores'}
]
pd.DataFrame(architecture_rows)

,nivel,modulo,responsabilidad
0,Config,app/config/settings.py,Lee config.yaml y expone datasets y rutas
1,Clients,app/clients/download_client.py,Descarga archivos directos
2,Clients,app/clients/mef_client.py,Pagina recursos MEF expuestos por datastore_se...
3,Storage,app/storage/data_lake.py,Escribe CSV en Bronze
4,Service,app/services/ingestion_service.py,"Coordina descarga, escritura y quality checks"
5,Pipelines,app/pipelines/bronze/*.py,Orquestación por dataset y pipeline maestro
6,Audit,app/audit/*.py,"Registra ejecuciones, quality checks y errores"


## Ejecución del pipeline

La siguiente celda permite ejecutar el pipeline completo. Por defecto no vuelve a ejecutarlo para evitar descargas innecesarias. Si quieres volver a correr la ingesta, cambia `RUN_PIPELINE` a `True`.

Si no se ejecuta, el notebook toma el último log de auditoría disponible para mostrar el resultado más reciente.

In [4]:
from app.pipelines.bronze.bronze_pipeline import BronzePipeline

RUN_PIPELINE = False

if RUN_PIPELINE:
    pipeline = BronzePipeline()
    pipeline_result = pipeline.run()
else:
    pipeline_result = None

pipeline_result

In [5]:
execution_files = sorted((AUDIT_PATH / 'executions').glob('*/*/*/*.json'))
latest_execution_file = execution_files[-1] if execution_files else None

if latest_execution_file is None:
    raise FileNotFoundError('No se encontró ningún log de ejecución en data/audit/executions')

with open(latest_execution_file, 'r', encoding='utf-8') as file_handle:
    latest_execution = json.load(file_handle)

latest_execution_file, latest_execution

(PosixPath('/home/jovyan/work/data/audit/executions/2026/05/21/20260521_235615_506598.json'),
 {'audit_id': '20260521_235615_506598',
  'pipeline_name': 'bronze_partial_ingestion',
  'execution_id': '20260521_235530',
  'timestamp': '2026-05-21T23:56:15.506616',
  'status': 'partial',
  'duration_seconds': 44.715337,
  'records_processed': 32,
  'records_success': 31,
  'records_failed': 1,
  'error_details': [{'dataset': 'sismepre',
    'asset_name': 'rentas_entidad_estado_diccionario.csv',
    'error': 'No records returned for resource URL: https://api.datosabiertos.mef.gob.pe/DatosAbiertos/v1/datastore_search?resource_id=f2d375f8-9f67-4bf8-849f-9082e7a6eb89'}],
  'metadata': {'datasets': ['ingresos', 'sismepre', 'renamu']}})

## Evidencia de archivos Bronze

Esta sección comprueba qué archivos están presentes en la capa Bronze por dataset y muestra tamaño y fecha de modificación. Es una evidencia útil para la defensa del laboratorio porque conecta el pipeline con el data lake resultante.

In [6]:
bronze_rows = []
for dataset_dir in sorted(BRONZE_PATH.iterdir()):
    if not dataset_dir.is_dir():
        continue
    for file_path in sorted(dataset_dir.iterdir()):
        if file_path.is_file():
            bronze_rows.append({
                'dataset': dataset_dir.name,
                'filename': file_path.name,
                'size_mb': round(file_path.stat().st_size / (1024 * 1024), 2),
                'modified': pd.Timestamp(file_path.stat().st_mtime, unit='s')
            })

bronze_df = pd.DataFrame(bronze_rows).sort_values(['dataset', 'filename']).reset_index(drop=True)
bronze_df

,dataset,filename,size_mb,modified
0,ingresos,2012-Ingreso.csv,3.75,2026-05-15 17:05:37.580126762
1,ingresos,2013-Ingreso.csv,7.25,2026-05-15 17:05:37.611871004
2,ingresos,2014-Ingreso.csv,3.75,2026-05-15 17:05:37.633887529
3,ingresos,2015-Ingreso.csv,3.75,2026-05-15 17:05:37.657779455
4,ingresos,2016-Ingreso.csv,3.75,2026-05-15 17:05:37.679910898
5,ingresos,2017-Ingreso.csv,3.75,2026-05-15 17:05:37.704187632
6,ingresos,2018-Ingreso.csv,3.75,2026-05-15 17:05:37.727518320
7,ingresos,2019-Ingreso.csv,3.75,2026-05-15 17:05:37.750867367
8,ingresos,2020-Ingreso.csv,3.75,2026-05-15 17:05:37.774036407
9,ingresos,2021-Ingreso.csv,3.75,2026-05-15 17:05:37.796814919


In [7]:
bronze_df.groupby('dataset').agg(total_files=('filename', 'count'), total_size_mb=('size_mb', 'sum')).reset_index()

,dataset,total_files,total_size_mb
0,ingresos,16,59.75
1,renamu,4,6.96
2,sismepre,13,28.98


## Evidencia de auditoría

El sistema de auditoría guarda tres tipos de evidencia:

- `executions`: resumen de la corrida.
- `quality_checks`: controles mínimos ejecutados sobre Bronze.
- `errors`: incidentes encontrados durante la ingesta.

A continuación se muestran los archivos más recientes y el contenido resumido de la última ejecución.

In [8]:
quality_files = sorted((AUDIT_PATH / 'quality_checks').glob('*/*/*/*.json'))
error_files = sorted((AUDIT_PATH / 'errors').glob('*/*/*/*.json'))

audit_summary = {
    'latest_execution_file': str(latest_execution_file),
    'quality_checks_count': len(quality_files),
    'error_files_count': len(error_files),
    'latest_error_file': str(error_files[-1]) if error_files else None
}
audit_summary

{'latest_execution_file': '/home/jovyan/work/data/audit/executions/2026/05/21/20260521_235615_506598.json',
 'quality_checks_count': 39,
 'error_files_count': 1,
 'latest_error_file': '/home/jovyan/work/data/audit/errors/2026/05/21/error_20260521_235531_877154.json'}

In [9]:
execution_report_df = pd.DataFrame([latest_execution])
execution_report_df.T

,0
audit_id,20260521_235615_506598
pipeline_name,bronze_partial_ingestion
execution_id,20260521_235530
timestamp,2026-05-21T23:56:15.506616
status,partial
duration_seconds,44.715337
records_processed,32
records_success,31
records_failed,1
error_details,"[{'dataset': 'sismepre', 'asset_name': 'rentas..."


## Incidencias detectadas

Esta sección documenta los recursos que quedaron con error o advertencia. En tu caso, el pipeline quedó operativo y solo se detectó una incidencia puntual asociada a un recurso API de SISMEPRE que respondió sin registros.

In [10]:
errors = latest_execution.get('error_details') or []
pd.DataFrame(errors) if errors else pd.DataFrame(columns=['dataset', 'asset_name', 'error'])

,dataset,asset_name,error
0,sismepre,rentas_entidad_estado_diccionario.csv,No records returned for resource URL: https://...


## Conclusiones

- El pipeline Bronze del parcial ya aterriza las tres fuentes en `data/bronze`.
- El flujo es idempotente: si el archivo ya existe, se registra como `skipped_existing`.
- La auditoría deja evidencia suficiente para sustentar la ejecución del laboratorio.
- La siguiente fase del trabajo no es Silver ni Gold, sino el profiling y la evaluación de calidad de los datos ya cargados en Bronze.